# Lab | LangChain Med

## Objectives

- continue on with lesson 2' example, use different datasets to test what we did in class. Some datasets are suggested in the notebook but feel free to scout other datasets on HuggingFace or Kaggle.
- Find another model on Hugging Face and compare it.
- Modify the prompt to fit your selected dataset.

In [3]:
import numpy as np 
import pandas as pd

## Load the Dataset
As you can see the notebook is ready to work with three different Datasets. Just uncomment the lines of the Dataset you want to use. 

I selected Datasets with News. Two of them have just a brief decription of the news, but the other contains the full text. 

As we are working in a free and limited space, I limited the number of news to use with the variable MAX_NEWS. Feel free to pull more if you have memory available. 

The name of the field containing the text of the new is stored in the variable *DOCUMENT* and the metadata in *TOPIC*

In [4]:
# news = pd.read_csv('/kaggle/input/topic-labeled-news-dataset/labelled_newscatcher_dataset.csv', sep=';')
# MAX_NEWS = 1000
# DOCUMENT="title"
# TOPIC="topic"

news = pd.read_csv('/Users/test/Desktop/ironhack_labs/lab-langchain-med/datasets/bbc_news.csv')
MAX_NEWS = 1000
DOCUMENT="description"
TOPIC="title"

# news = pd.read_csv('/kaggle/input/mit-ai-news-published-till-2023/articles.csv')
# MAX_NEWS = 100
# DOCUMENT="Article Body"
# TOPIC="Article Header"

# news = "PICK A DATASET" #Ideally pick one from the commented ones above

print(type(news))

<class 'pandas.core.frame.DataFrame'>


ChromaDB requires that the data has a unique identifier. We can make it with this statement, which will create a new column called **Id**.


In [5]:
news["id"] = news.index
news.head()

,title,pubDate,guid,link,description,id
0,Ukraine: Angry Zelensky vows to punish Russian...,"Mon, 07 Mar 2022 08:01:56 GMT",https://www.bbc.co.uk/news/world-europe-60638042,https://www.bbc.co.uk/news/world-europe-606380...,The Ukrainian president says the country will ...,0
1,War in Ukraine: Taking cover in a town under a...,"Sun, 06 Mar 2022 22:49:58 GMT",https://www.bbc.co.uk/news/world-europe-60641873,https://www.bbc.co.uk/news/world-europe-606418...,"Jeremy Bowen was on the frontline in Irpin, as...",1
2,Ukraine war 'catastrophic for global food',"Mon, 07 Mar 2022 00:14:42 GMT",https://www.bbc.co.uk/news/business-60623941,https://www.bbc.co.uk/news/business-60623941?a...,One of the world's biggest fertiliser firms sa...,2
3,Manchester Arena bombing: Saffie Roussos's par...,"Mon, 07 Mar 2022 00:05:40 GMT",https://www.bbc.co.uk/news/uk-60579079,https://www.bbc.co.uk/news/uk-60579079?at_medi...,The parents of the Manchester Arena bombing's ...,3
4,Ukraine conflict: Oil price soars to highest l...,"Mon, 07 Mar 2022 08:15:53 GMT",https://www.bbc.co.uk/news/business-60642786,https://www.bbc.co.uk/news/business-60642786?a...,Consumers are feeling the impact of higher ene...,4


In [6]:
#Because it is just a course we select a small portion of News.
subset_news = news.head(MAX_NEWS)

print(subset_news)

                                                 title  \
0    Ukraine: Angry Zelensky vows to punish Russian...   
1    War in Ukraine: Taking cover in a town under a...   
2           Ukraine war 'catastrophic for global food'   
3    Manchester Arena bombing: Saffie Roussos's par...   
4    Ukraine conflict: Oil price soars to highest l...   
..                                                 ...   
995  Ukraine: Premature babies struggling for life ...   
996  Russia transfers thousands of Mariupol civilia...   
997  In Chernihiv, civilians are trapped as hospita...   
998  Russian oligarchs: Where do they hide their 'd...   
999  Russian agent linked to assassination team was...   

                           pubDate  \
0    Mon, 07 Mar 2022 08:01:56 GMT   
1    Sun, 06 Mar 2022 22:49:58 GMT   
2    Mon, 07 Mar 2022 00:14:42 GMT   
3    Mon, 07 Mar 2022 00:05:40 GMT   
4    Mon, 07 Mar 2022 08:15:53 GMT   
..                             ...   
995  Sun, 27 Mar 2022 23:15:27 GMT   

## Import and configure the Vector Database
I'm going to use ChromaDB, the most popular OpenSource embedding Database. 

First we need to import ChromaDB, and after that import the **Settings** class from **chromadb.config** module. This class allows us to change the setting for the ChromaDB system, and customize its behavior. 

In [7]:
!pip install chromadb

In [8]:
import chromadb
from chromadb.config import Settings

To create the settings_chroma object, we call the Settings function that was previously imported. This object requires two parameters:

chroma_db_impl: This defines the database implementation and the format used to store the data. We choose duckdb+parquet, which offers high performance by operating primarily in memory, is fully compatible with SQL, and uses the Parquet format — ideal for tabular data due to its efficient compression and performance.

persist_directory: This specifies the directory where the data will be saved. While it's possible to run without persisting data (storing it only in memory), Kaggle does not support non-persistent storage, so we must provide a directory.

<!-- Now we need to create the seetings object calling the **Settings** function imported previously. We store the object in the variable **settings_chroma**.

Is necessary to inform two parameters 
* chroma_db_impl. Here we specify the database implementation and the format how store the data. I choose ***duckdb***, because his high-performace. It operate primarly in memory. And is fully compatible with SQL. The store format ***parquet*** is good for tabular data. With good compression rates and performance. 

* persist_directory: It just contains the directory where the data will be stored. Is possible work without a directory and the data will be stored in memory without persistece, but Kaggle dosn't support that.  -->

In [9]:
# settings_chroma = Settings

from chromadb.config import Settings

settings_chroma = Settings(
    chroma_db_impl="duckdb+parquet",
    persist_directory="/kaggle/working/chroma_data"
)

chroma_client = chromadb.PersistentClient(path="./chroma_data")

## Filling and Querying the ChromaDB Database
The Data in ChromaDB is stored in collections. If the collection exist we need to delete it. 

In the next lines, we are creating the collection by calling the ***create_collection*** function in the ***chroma_client*** created above.

In [10]:
collection_name = "news_collection"
if len(chroma_client.list_collections()) > 0 and collection_name in [chroma_client.list_collections()[0].name]:
        chroma_client.delete_collection(name=collection_name)

collection = chroma_client.create_collection(name=collection_name)
    

It's time to add the data to the collection. Using the function ***add*** we need to inform, at least ***documents***, ***metadatas*** and ***ids***. 
* In the **document** we store the big text, it's a different column in each Dataset. 
* In **metadatas**, we can informa a list of topics. 
* In **id** we need to inform an unique identificator for each row. It MUST be unique! I'm creating the ID using the range of MAX_NEWS. 


In [12]:
# collection.add(
#     documents=subset_news[DOCUMENT].tolist()[:100],
#     metadatas=[{TOPIC: topic} for topic in subset_news[TOPIC].tolist()[:100]],
#     ids=[f"id{x}" for x in range(100)],
# )

# collection.add(
#     documents=subset_news[DOCUMENT].tolist(),
#     metadatas=[{TOPIC: topic} for topic in subset_news[TOPIC].tolist()],
#     ids=[f"id{x}" for x in range(MAX_NEWS)],
# )

!pip install -q sentence-transformers
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')  # small and fast
embeddings = model.encode(subset_news[DOCUMENT].tolist()[:100])

collection.add(
    documents=subset_news[DOCUMENT].tolist()[:100],
    metadatas=[{TOPIC: topic} for topic in subset_news[TOPIC].tolist()[:100]],
    ids=[f"id{x}" for x in range(100)],
    embeddings=embeddings
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [13]:
results = collection.query(query_texts=["laptop"], n_results=10 )

print(results)

{'ids': [['id7', 'id32', 'id21', 'id36', 'id13', 'id51', 'id28', 'id65', 'id64', 'id67']], 'embeddings': None, 'documents': [['TikTok suspends live streaming and new content from its platform, while Russians can no longer access Netflix.', "A student at Queen's University Belfast used his daily walks to record the sounds of Covid lockdowns.", 'The video, shared by Ukrainian armed forces, shows the helicopter being hit by a missile.', 'Ad-testing firms are using real-time software, webcams to find out if ads resonate with customers.', "Ukraine's youngest cabinet minister, 31-year-old Mykhailo Fedorov, is leading a cyber-battle against Russia.", 'Explore the data on coronavirus in the UK and find out how many cases there are in your area.', 'Lesya and Valeri, both civil defence volunteers, tied the knot at a checkpoint near Kyiv.', 'The family were posing for a photo when a shooting star shot through the background.', 'A 19-year-old Irish woman is among hundreds of international students

## Vector MAP

In [15]:
!pip install matplotlib scikit-learn

import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


  Using cached matplotlib-3.10.3-cp310-cp310-macosx_11_0_arm64.whl.metadata (11 kB)
  Using cached contourpy-1.3.2-cp310-cp310-macosx_11_0_arm64.whl.metadata (5.5 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached kiwisolver-1.4.8-cp310-cp310-macosx_11_0_arm64.whl.metadata (6.2 kB)
  Using cached pyparsing-3.2.3-py3-none-any.whl.metadata (5.0 kB)
Using cached matplotlib-3.10.3-cp310-cp310-macosx_11_0_arm64.whl (8.0 MB)
Using cached contourpy-1.3.2-cp310-cp310-macosx_11_0_arm64.whl (253 kB)
Using cached cycler-0.12.1-py3-none-any.whl (8.3 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 5.3 MB/s eta 0:00:00a 0:00:01
Using cached kiwisolver-1.4.8-cp310-cp310-macosx_11_0_arm64.whl (65 kB)
Using cached pyparsing-3.2.3-py3-none-any.whl (111 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6/6 [matplotlib]6 [matplotlib]


In [16]:

getado = collection.get(ids="id141", 
                       include=["documents", "embeddings"])


In [17]:
word_vectors = getado["embeddings"]
word_list = getado["documents"]
word_vectors

array([], dtype=float64)

Once we have our information inside the Database we can query It, and ask for data that matches our needs. The search is done inside the content of the document, and it dosn't look for the exact word, or phrase. The results will be based on the similarity between the search terms and the content of documents. 

The metadata is not used in the search, but they can be utilized for filtering or refining the results after the initial search. 


## Loading the model and creating the prompt
TRANSFORMERS!!
Time to use the library **transformers**, the most famous library from [hugging face](https://huggingface.co/) for working with language models. 

We are importing: 
* **Autotokenizer**: It is a utility class for tokenizing text inputs that are compatible with various pre-trained language models.
* **AutoModelForCasualLLM**: it provides an interface to pre-trained language models specifically designed for language generation tasks using causal language modeling (e.g., GPT models), or the model used in this notebook ***databricks/dolly-v2-3b***.
* **pipeline**: provides a simple interface for performing various natural language processing (NLP) tasks, such as text generation (our case) or text classification. 

The model selected is [dolly-v2-3b](https://huggingface.co/databricks/dolly-v2-3b), the smallest Dolly model. It have 3billion paramaters, more than enough for our sample, and works much better than GPT2. 

Please, feel free to test [different Models](https://huggingface.co/models?pipeline_tag=text-generation&sort=trending), you need to search for NLP models trained for text-generation. My recomendation is choose "small" models, or we will run out of memory in kaggle.  


In [18]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

model_id = "databricks/dolly-v2-3b"
tokenizer = AutoTokenizer.from_pretrained(model_id)
lm_model = AutoModelForCausalLM.from_pretrained(model_id)



tokenizer_config.json:   0%|          | 0.00/450 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.11M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/228 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/819 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/5.68G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/5.68G [00:00<?, ?B/s]

The next step is to initialize the pipeline using the objects created above. 

The model's response is limited to 256 tokens, for this project I'm not interested in a longer response, but it can easily be extended to whatever length you want.

Setting ***device_map*** to ***auto*** we are instructing the model to automaticaly select the most appropiate device: CPU or GPU for processing the text generation.  

In [20]:
pipe = pipeline(
    "text-generation",
    model=lm_model,
    tokenizer=tokenizer,
    max_new_tokens=256,
    device_map="auto",
)

Device set to use mps:0


## Creating the extended prompt
To create the prompt we use the result from query the Vector Database  and the sentence introduced by the user. 

The prompt have two parts, the **relevant context** that is the information recovered from the database and the **user's question**. 

We only need to join the two parts together to create the prompt that we are going to send to the model. 

You can limit the lenght of the context passed to the model, because we can get some Memory problems with one of the datasets that contains a realy large text in the document part. 

In [21]:
question = "Can I buy a Toshiba laptop?"
context = " ".join([f"#{str(i)}" for i in results["documents"][0]])
#context = context[0:5120]
prompt_template = f"Relevant context: {context}\n\n The user's question: {question}"
prompt_template

"Relevant context: #TikTok suspends live streaming and new content from its platform, while Russians can no longer access Netflix. #A student at Queen's University Belfast used his daily walks to record the sounds of Covid lockdowns. #The video, shared by Ukrainian armed forces, shows the helicopter being hit by a missile. #Ad-testing firms are using real-time software, webcams to find out if ads resonate with customers. #Ukraine's youngest cabinet minister, 31-year-old Mykhailo Fedorov, is leading a cyber-battle against Russia. #Explore the data on coronavirus in the UK and find out how many cases there are in your area. #Lesya and Valeri, both civil defence volunteers, tied the knot at a checkpoint near Kyiv. #The family were posing for a photo when a shooting star shot through the background. #A 19-year-old Irish woman is among hundreds of international students taking cover in Sumy. #Lesya and Valeri, both civil defence volunteers, tied the knot at a check-point near Kyiv.\n\n The 

Now all that remains is to send the prompt to the model and wait for its response!


In [23]:
lm_response = pipe(prompt_template)
print(lm_response[0]["generated_text"])

RuntimeError: MPS backend out of memory (MPS allocated: 9.06 GB, other allocations: 3.00 MB, max allowed: 9.07 GB). Tried to allocate 2.40 MB on private pool. Use PYTORCH_MPS_HIGH_WATERMARK_RATIO=0.0 to disable upper limit for memory allocations (may cause system failure).